# Optimize Weld's Experiment with Ax


In [55]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig

In [56]:
NUM_NEW_EXP = 10

In [57]:
# 1. Initialize the Client.
client = Client(random_seed=42)

In [58]:
folder_path = Path(os.getcwd()) / "data"

In [59]:
exp = pd.read_excel(folder_path / "experiments.xlsx")
display(exp)


,trial_index,R1,R2,R3,W1,W2,W3,D,visual_quality,sigma_advancing_1,sigma_advancing_2,sigma_advancing_3,sigma_retreating_1,sigma_retreating_2,sigma_retreating_3


In [60]:
print(exp.columns)
cols_advancing = [
    "sigma_advancing_1",
    "sigma_advancing_2",
    "sigma_advancing_3",
]
cols_retreating = [
    "sigma_retreating_1",
    "sigma_retreating_2",
    "sigma_retreating_3",
]

Index(['trial_index', 'R1', 'R2', 'R3', 'W1', 'W2', 'W3', 'D',
       'visual_quality', 'sigma_advancing_1', 'sigma_advancing_2',
       'sigma_advancing_3', 'sigma_retreating_1', 'sigma_retreating_2',
       'sigma_retreating_3'],
      dtype='str')


In [61]:
filt_advancing = exp[cols_advancing].isna().all(axis=1)
filt_retreating = exp[cols_retreating].isna().all(axis=1)
filt_removed = filt_advancing | filt_retreating
filt_removed

Series([], dtype: bool)

In [62]:
exp_todo = exp[filt_removed].copy()
exp = exp[~filt_removed].copy()
display(exp)
display(exp_todo)

,trial_index,R1,R2,R3,W1,W2,W3,D,visual_quality,sigma_advancing_1,sigma_advancing_2,sigma_advancing_3,sigma_retreating_1,sigma_retreating_2,sigma_retreating_3


,trial_index,R1,R2,R3,W1,W2,W3,D,visual_quality,sigma_advancing_1,sigma_advancing_2,sigma_advancing_3,sigma_retreating_1,sigma_retreating_2,sigma_retreating_3


In [63]:
# 2. Configure where Ax will search.

bound_R = [1400, 2000]  # Rotatinal speed
bound_W = [40, 100]  # Welding speed
bound_D = [10, 30]  # Dwell time

parameters = [
    RangeParameterConfig(
        name="R1",
        bounds=bound_R,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="R2",
        bounds=bound_R,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="R3",
        bounds=bound_R,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="W1",
        bounds=bound_W,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="W2",
        bounds=bound_W,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="W3",
        bounds=bound_W,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="D",
        bounds=bound_D,
        parameter_type="int",
    ),
]

client.configure_experiment(
    name="parn_exp",
    parameters=parameters,
)

In [64]:
client.configure_optimization(
    objective="sigma_advancing, sigma_retreating",
    outcome_constraints=["visual_quality >= 0.5"],
)

In [65]:
def cal_mean_sem(row, cols):
    filt = row[cols].notna()
    arr = row[cols][filt].astype(float).values
    mean = arr.mean()
    sem = arr.std() / np.sqrt(len(arr))
    return mean, sem

In [66]:
for idx, row in exp.iterrows():
    trial_index = client.attach_trial(
        parameters={
            "R1": int(row["R1"]),
            "R2": int(row["R2"]),
            "R3": int(row["R3"]),
            "W1": int(row["W1"]),
            "W2": int(row["W2"]),
            "W3": int(row["W3"]),
            "D": int(row["D"]),
        }
    )
    advancing_mean, advancing_sem = cal_mean_sem(row, cols_advancing)
    retreating_mean, retreating_sem = cal_mean_sem(row, cols_retreating)
    visual_quality = float(row["visual_quality"])
    print(f"Trial {trial_index}: mean={advancing_mean}, sem={advancing_sem}")
    print(f"Trial {trial_index}: mean={retreating_mean}, sem={retreating_sem}")
    print(f"Trial {trial_index}: visual_quality={visual_quality}")
    client.complete_trial(
        trial_index=trial_index,
        raw_data={
            "sigma_advancing": (advancing_mean, advancing_sem),
            "sigma_retreating": (retreating_mean, retreating_sem),
            "visual_quality": visual_quality,
        },
    )

In [67]:
if exp_todo.shape[0] == 0:
    suggest_params = []
    try:
        # Request only the intended number of new trials once.
        data = client.get_next_trials(max_trials=NUM_NEW_EXP)
        for trial_index, parameters in data.items():
            print(f"Trial {trial_index} with parameters {parameters}")
            suggest_params.append({"trial_index": trial_index, **parameters})
    except Exception as e:
        if type(e).__name__ == "DataRequiredError":
            print(
                "Ax cannot generate more trials yet: more observed trial data is "
                "required before moving past Sobol."
            )
        else:
            raise

    if suggest_params:
        _exp = pd.DataFrame(suggest_params)
        _exp[cols_advancing + cols_retreating] = pd.NA
        exp = pd.concat([exp, _exp], ignore_index=True).reset_index(drop=True)
        display(exp)
        SAVE_EXCEL = True
    else:
        SAVE_EXCEL = False
else:
    print("There are still experiment to be conducted. Skip generating new sets")
    SAVE_EXCEL = False

[INFO 07-22 10:13:29] ax.api.client: GenerationStrategy(name='Center+Sobol+MBM:fast', nodes=[CenterGenerationNode(next_node_name='Sobol', use_existing_trials_for_initialization=True), GenerationNode(name='Sobol', generator_specs=[GeneratorSpec(generator_enum=Sobol, generator_key_override=None)], transition_criteria=[MinTrials(transition_to='MBM'), MinTrials(transition_to='MBM')], suggested_experiment_status=ExperimentStatus.INITIALIZATION, pausing_criteria=[MaxTrialsAwaitingData(threshold=5)]), GenerationNode(name='MBM', generator_specs=[GeneratorSpec(generator_enum=BoTorch, generator_key_override=None)], transition_criteria=None, suggested_experiment_status=ExperimentStatus.OPTIMIZATION, pausing_criteria=None)]) chosen based on user input and problem structure.
[INFO 07-22 10:13:29] ax.api.client: Generated new trial 0 with parameters {'R1': 1700, 'R2': 1700, 'R3': 1700, 'W1': 70, 'W2': 70, 'W3': 70, 'D': 20} using GenerationNode CenterOfSearchSpace.
[INFO 07-22 10:13:29] ax.api.clien

Trial 0 with parameters {'R1': 1700, 'R2': 1700, 'R3': 1700, 'W1': 70, 'W2': 70, 'W3': 70, 'D': 20}
Trial 1 with parameters {'R1': 1462, 'R2': 1811, 'R3': 1851, 'W1': 93, 'W2': 63, 'W3': 41, 'D': 17}
Trial 2 with parameters {'R1': 1821, 'R2': 1598, 'R3': 1421, 'W1': 62, 'W2': 74, 'W3': 99, 'D': 20}
Trial 3 with parameters {'R1': 1894, 'R2': 1873, 'R3': 1756, 'W1': 73, 'W2': 98, 'W3': 82, 'D': 14}
Trial 4 with parameters {'R1': 1691, 'R2': 1519, 'R3': 1622, 'W1': 51, 'W2': 41, 'W3': 57, 'D': 29}
Trial 5 with parameters {'R1': 1561, 'R2': 1943, 'R3': 1492, 'W1': 42, 'W2': 48, 'W3': 47, 'D': 12}
Trial 6 with parameters {'R1': 1957, 'R2': 1466, 'R3': 1959, 'W1': 83, 'W2': 90, 'W3': 92, 'D': 26}
Trial 7 with parameters {'R1': 1730, 'R2': 1704, 'R3': 1664, 'W1': 64, 'W2': 83, 'W3': 75, 'D': 19}
Trial 8 with parameters {'R1': 1489, 'R2': 1687, 'R3': 1835, 'W1': 92, 'W2': 57, 'W3': 66, 'D': 23}
Trial 9 with parameters {'R1': 1549, 'R2': 1920, 'R3': 1641, 'W1': 78, 'W2': 74, 'W3': 59, 'D': 11}


,trial_index,R1,R2,R3,W1,W2,W3,D,visual_quality,sigma_advancing_1,sigma_advancing_2,sigma_advancing_3,sigma_retreating_1,sigma_retreating_2,sigma_retreating_3
0,0,1700,1700,1700,70,70,70,20,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,1,1462,1811,1851,93,63,41,17,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,2,1821,1598,1421,62,74,99,20,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,3,1894,1873,1756,73,98,82,14,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,4,1691,1519,1622,51,41,57,29,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,5,1561,1943,1492,42,48,47,12,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,6,1957,1466,1959,83,90,92,26,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,7,1730,1704,1664,64,83,75,19,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
8,8,1489,1687,1835,92,57,66,23,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
9,9,1549,1920,1641,78,74,59,11,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [68]:
if SAVE_EXCEL:
    exp.to_excel(folder_path / "experiments.xlsx", index=False)